# Step and Sinusoidal Responses of STD Sensitivity Functions

Generates Fig. S1 (`STP-response.pdf`). This notebook shows the time-domain response of the STD sensitivity functions and effective presynaptic terms used in the weak-coupling learning rule, using step and sinusoidal presynaptic-rate inputs.


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.integrate import solve_ivp
from scipy import signal

from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURE_DIR = ROOT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Shared plotting style for manuscript PDFs.
config = {
    "font.family": "sans-serif",
    "font.size": 18.0,
    "axes.titlelocation": "left",
    "axes.titlesize": 19.0,
    "axes.labelsize": 19.0,
    "xtick.labelsize": 17.0,
    "ytick.labelsize": 17.0,
    "legend.fontsize": 17.0,
    "figure.titlesize": 19.0,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    "lines.markersize": 4.0,
    "patch.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "axes.xmargin": 0.01,
    "axes.ymargin": 0.05,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.size": 2.0,
    "ytick.minor.size": 2.0,
    "xtick.minor.width": 0.6,
    "ytick.minor.width": 0.6,
    "legend.frameon": False,
    "legend.fancybox": False,
    "image.interpolation": "none",
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}
plt.rcParams.update(config)


In [ ]:
# Define activation functions
def activation_function_exp(u, params):
    """Exponential firing-rate nonlinearity."""
    return params['g_M'] * np.exp(params['beta'] * (u - params['u_c']))

def activation_function_sigmoid(u, params):
    """Sigmoid firing-rate nonlinearity."""
    from scipy.special import expit
    return params['g_M'] * expit(params['beta'] * (u - params['u_c']))

def dynamics_STP_factor(t, y, U, nu0_func, tau_d):
    '''
    dynamics of STP factor.
        w: synaptic efficacy
        f_w0: derivative with respect to w0
        f_U: derivative with respect to U
        U: release prob.
        nu0_func: pre firing rate (as func of time)
        tau_d: STD time constant
    '''
    f_w0, f_U = y
    nu0_t = nu0_func(t)

    recovery = 1 / tau_d
    usage = nu0_t * U

    df_w0_dt = recovery * (1-f_w0) - usage * f_w0
    df_U_dt = recovery * (1 - f_U) - usage * (f_U + f_w0)
    return [df_w0_dt, df_U_dt]


In [ ]:
params = {
    'freq': 1.0,
    'amp': 10.0,
    'tau_d': 0.5,  # Depression time constant (s)
    'beta' : 2.0,  # Steepness of activation function
    'g_M' : 10.0, # Maximum firing rate (Hz) for sigmoid activation.
    'u_c' : 1.0,    # Activation threshold
    'tau_s' : 0.01,  # Synaptic time constant (s),
    'U_init' : 0.15,
    'w0_init' : 1.0,
    'activation' : 'exp',
}


In [ ]:
# response to step function
def step_func(t, amp, t_start, t_end):
    """Step function: 0 -> amp Hz at t=t_start"""
    return np.where((t > t_start) & (t < t_end), amp, 0.0)
def sin_func(t, base, amp, freq):
    """Sinusoidal function: base + amp * sin(2pi * freq * t)"""
    return base + amp * np.sin(2.0 * np.pi * freq * t)

t0 = -1.0
t1 = 4.0
t_eval = np.linspace(t0, t1, 1000)

nu_step = lambda t: step_func(t, params['amp'], 0.0, 2.0)
nu_sin = lambda t: sin_func(t, params['amp'], params['amp'], params['freq'])
sol_step = solve_ivp(
    dynamics_STP_factor,
    [t0, t1],
    [1.0, 1.0], # initial conditions for f_w0 and f_U
    t_eval=t_eval,
    args=(params['U_init'], nu_step, params['tau_d']),
    method='BDF',
    max_step=0.01,
)
sol_sin = solve_ivp(
    dynamics_STP_factor,
    [t0, t1],
    [1.0, 1.0], # initial conditions for f_w0 and f_U
    t_eval=t_eval,
    args=(params['U_init'], nu_sin, params['tau_d']),
    method='BDF',
    max_step=0.01,
)

f_w0_step, f_U_step = sol_step.y
f_w0_sin, f_U_sin = sol_sin.y


fig, axes = plt.subplots(3, 2, figsize=(12.0, 6.0), sharex = True, sharey = 'row', constrained_layout=True)

# plot step input
ax = axes[0,0]
ax.plot(t_eval, nu_step(t_eval), color='black')
ax.set_title('A')
ax.set_ylabel('Firing rate \n'+ r'$\nu (t)$')
ax.set_ylim(-1, 21)

# plot step response
ax = axes[1,0]
ax.plot(t_eval, f_w0_step, label=r'$f^{w_0}(t)$', color='C0')
ax.plot(t_eval, f_U_step, label=r'$f^{U}(t)$', color='C1')
ax.set_title('B')
ax.set_ylabel('Sensitivity \n' + r'$f^Z(t)$')
ax.set_ylim(-0.1, 1.1)
# ax.legend()

ax = axes[2,0]
ax.plot(t_eval, nu_step(t_eval) * f_w0_step, label=r'$\nu(t) \cdot f^{w_0}(t)$', color='C0')
ax.plot(t_eval, nu_step(t_eval) * f_U_step, label=r'$\nu(t) \cdot f^{U}(t)$', color='C1')
ax.set_title('C')
ax.set_ylabel('Presynaptic \n component \n'+ r'$C^Z(t)$')
# ax.legend()

# plot sinusoidal input
ax = axes[0,1]
ax.plot(t_eval, nu_sin(t_eval), color='black')
ax.set_title('D')

# plot sinusoidal response
ax = axes[1,1]
ax.plot(t_eval, f_w0_sin, label=r'$f^{w_0}(t)$', color='C0')
ax.plot(t_eval, f_U_sin, label=r'$f^{U}(t)$', color='C1')
ax.set_title('E')
# ax.legend()

ax = axes[2,1]
ax.plot(t_eval, nu_sin(t_eval) * f_w0_sin, label=r'$\nu(t) \cdot f^{w_0}(t)$', color='C0')
ax.plot(t_eval, nu_sin(t_eval) * f_U_sin, label=r'$\nu(t) \cdot f^{U}(t)$', color='C1')
ax.set_title('F')
# ax.legend()

legend_handles = [
    Line2D([0], [0], color='C0', lw=1.8, label=r'$w_0$'),
    Line2D([0], [0], color='C1', lw=1.8, label=r'$U$'),
]
fig.legend(
    handles=legend_handles,
    loc='center left',
    bbox_to_anchor=(1.01, 0.5),
    borderaxespad=0.0,
)

for ax in axes.flatten():
    ax.grid()
for ax in axes[2,:]:
    ax.set_xlabel('Time (s)')
plt.savefig(FIGURE_DIR / 'STP-response.pdf', bbox_inches='tight')
plt.show()

print("parameters:", params)
